ASG Airlines(Case Study) by Yamini D

I have created an End-to-end pipeline for the ASG Airlines: raw booking/scheduling/airport-log data goes through Bronze to Silver to Gold with a quarantine table for anything that fails validation along the way instead of deleting so that we can reprocess it later if needed

Cell 1: Setup and ingestion

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

FILE_PATH = "/Volumes/workspace/default/airline_data/UseCase - Airlines.xlsx"

#I am taking raw dataset as bronze to differentiate the levels(just for understanding the flow)
flights_raw    = pd.read_excel(FILE_PATH, sheet_name="flights")
bookings_raw   = pd.read_excel(FILE_PATH, sheet_name="bookings")
passengers_raw = pd.read_excel(FILE_PATH, sheet_name="passengers")
payments_raw   = pd.read_excel(FILE_PATH, sheet_name="payments")

print("Bronze row counts:")
for name, df in [("flights", flights_raw), ("bookings", bookings_raw),
                  ("passengers", passengers_raw), ("payments", payments_raw)]:
    print(f"  {name:12s}: {len(df)} rows, {df.shape[1]} columns")

Bronze row counts:
  flights     : 1020 rows, 7 columns
  bookings    : 1000 rows, 9 columns
  passengers  : 1039 rows, 9 columns
  payments    : 1000 rows, 4 columns


Cell 2: Profiling(duplicates and nulls)

In [0]:
def profile_table(df, name):
    print(f"\n{name}")
    print("Null counts:\n", df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())

for name, df in [("flights", flights_raw), ("bookings", bookings_raw),("passengers", passengers_raw), ("payments", payments_raw)]:
    profile_table(df, name)


flights
Null counts:
 flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64
Duplicate rows: 15

bookings
Null counts:
 booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
Duplicate rows: 0

passengers
Null counts:
 passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64
Duplicate rows: 0

payments
Null counts:
 payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
Duplicate rows: 0


Data Type Check

In [0]:
print(flights_raw.dtypes)
print(bookings_raw.dtypes)
print(passengers_raw.dtypes)
print(payments_raw.dtypes)

flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email                    object
phone                    object
aadhaar_id                int64
date_of_birth    datetime64[ns]
dtype: object
payment_id        object
booking_id        object
amount     

fixing payment as numeric instead of object nad aadhar_id as string instead of int64

In [0]:

payments_raw["amount"] = pd.to_numeric(payments_raw["amount"],errors="coerce")

passengers_raw["aadhaar_id"] = (passengers_raw["aadhaar_id"].astype("string").str.strip())

bookings_raw["status"] = (bookings_raw["status"].astype("string").str.strip().str.upper())

Cell 3: Quarantine

Instead of deleting bad rows, I set them aside in a quarantine table. each one tagged with why it failed and which run caught it, so that way nothing is lost and I can go back and reprocess these later if needed

In [0]:
from datetime import datetime

pipeline_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
quarantine_records = []

def quarantine(df, source_table, error_code, error_reason):
    if df.empty:
        return

    q = df.copy()
    q["source_table"] = source_table
    q["error_code"] = error_code
    q["error_reason"] = error_reason

    quarantine_records.append(q)

Cell 4: validate flights

I did 4 checks here
1. I parsed both timestamps properly and quarantined any records that failed to parse
2. I standardized flight_id and validated it against the dataset format. Invalid IDs were quarantined
3. I handled overnight flights by rolling the arrival date forward when it was earlier than departure and the gap was under 12 hours. If it was still invalid, I quarantined it
4. I set is_overnight based on whether the arrival and departure fall on different calendar days

In [0]:
import re
flights_check = flights_raw.copy()

for col in ["departure_time", "arrival_time"]:
    flights_check[col] = pd.to_datetime(flights_check[col], errors="coerce")

missing_time = flights_check[flights_check["departure_time"].isna() | flights_check["arrival_time"].isna()]
quarantine(missing_time, "flights", "DQ002","departure_time or arrival_time could not be parsed")
flights_check = flights_check[~flights_check.index.isin(missing_time.index)]

flights_check["flight_id"] = flights_check["flight_id"].astype("string").str.strip().str.upper()
FLIGHT_ID_PATTERN = re.compile(r"^[A-Z0-9]{2}\d{3}$")
bad_flight_id = flights_check[~flights_check["flight_id"].str.match(FLIGHT_ID_PATTERN, na=False)]
quarantine(bad_flight_id, "flights", "DQ003", "flight_id doesn't match the [A-Z0-9]{2}+3-digit format")
flights_check = flights_check[~flights_check.index.isin(bad_flight_id.index)]

rollover_gap_hrs = (flights_check["departure_time"] - flights_check["arrival_time"]).dt.total_seconds() / 3600
plausible_rollover = (flights_check["arrival_time"] < flights_check["departure_time"]) & (rollover_gap_hrs <= 12)
flights_check.loc[plausible_rollover, "arrival_time"] += pd.Timedelta(days=1)

still_broken = flights_check[flights_check["arrival_time"] <= flights_check["departure_time"]]
quarantine(still_broken, "flights", "DQ004"," arrival_time not after departure_time, even after the overnight fix")
flights_check = flights_check[~flights_check.index.isin(still_broken.index)]

flights_check["is_overnight"] = (flights_check["departure_time"].dt.date != flights_check["arrival_time"].dt.date)
flights_check["computed_duration_min"] = (
    flights_check["arrival_time"] - flights_check["departure_time"]).dt.total_seconds() / 60

flights_valid = flights_check
print(f"flights: {len(flights_valid)} passed")
print("overnight flights:", flights_valid["is_overnight"].sum())

flights: 1019 passed
overnight flights: 124


Cell 5: validate bookings 
(referential-integrity check - I've explained it in word docx with examples) This is more like a constraint check which usually happens before implementing anything

In [0]:
bookings_check = bookings_raw.copy()

# Does every booking point to a flight that actually exists?
orphan_bookings = bookings_check[~bookings_check["flight_id"].isin(flights_raw["flight_id"])]
quarantine(orphan_bookings, "bookings","DQ005", "flight_id not found in flights table")

# Logical impossibility(booked after the flight already departed)
flight_departures = flights_valid[["flight_id", "departure_time"]].drop_duplicates("flight_id")
merged_check = bookings_check.merge(flight_departures, on="flight_id", how="left")
booked_too_late = merged_check[merged_check["booking_date"] > merged_check["departure_time"]]
quarantine(booked_too_late, "bookings", "DQ006" ,"booking_date after flight departure_time")

bookings_check["status"] = bookings_check["status"].replace({"INVALID": pd.NA}).fillna("UNKNOWN")

bad_booking_ids = set(orphan_bookings["booking_id"]) | set(booked_too_late["booking_id"])
bookings_valid = bookings_check[~bookings_check["booking_id"].isin(bad_booking_ids)]
print(f"bookings: {len(bookings_valid)} passed, {len(bad_booking_ids)} quarantined")

bookings: 1000 passed, 0 quarantined


Cell 6:validate payments


In [0]:
payments_check = payments_raw.copy()

orphan_payments = payments_check[~payments_check["booking_id"].isin(bookings_raw["booking_id"])]
quarantine(orphan_payments, "payments","DQ005", "booking_id not found among valid bookings")

bad_amount = payments_check[payments_check["amount"].isna() | (payments_check["amount"] <= 0)]
quarantine(bad_amount, "payments","DQ006", "amount is zero or negative or missing")

bad_payment_ids = set(orphan_payments["payment_id"]) | set(bad_amount["payment_id"])
payments_valid = payments_check[~payments_check["payment_id"].isin(bad_payment_ids)]
print(len(payments_valid), "passed,", len(bad_payment_ids), "quarantined")

922 passed, 78 quarantined


Cell 7: quarantine table

In [0]:

quarantine_table = pd.concat(quarantine_records, ignore_index=True) if quarantine_records else pd.DataFrame()
print("Total quarantined across flights/bookings/payments:", len(quarantine_table))
quarantine_table[["source_table", "error_code", "error_reason"]].value_counts()

Total quarantined across flights/bookings/payments: 79


source_table  error_code  error_reason                                                        
payments      DQ006       amount is zero or negative or missing                                   78
flights       DQ004        arrival_time not after departure_time, even after the overnight fix     1
Name: count, dtype: int64

cell 8: silver

In [0]:

#in passengers check aadhaar length and handle duplicate ids 
passengers_check = passengers_raw.copy()

valid_aadhaar = passengers_check["aadhaar_id"].str.fullmatch(r"\d{12}")
bad_aadhaar = passengers_check[~valid_aadhaar.fillna(False)]
quarantine(bad_aadhaar, "passengers", "DQ007", "aadhaar_id is not a valid 12-digit number")
passengers_check = passengers_check[valid_aadhaar.fillna(False)]

required_cols = ["first_name", "last_name", "age", "gender", "email", "phone", "aadhaar_id", "date_of_birth"]
passengers_check["_quality_score"] = passengers_check[required_cols].notna().sum(axis=1)
passengers_check = passengers_check.sort_values(["passenger_id", "_quality_score"], ascending=[True, False])

dup_mask = passengers_check["passenger_id"].duplicated(keep="first")
quarantine(passengers_check[dup_mask], "passengers", "DQ004", "duplicate passenger_id, kept the more complete record")
passengers_valid = passengers_check[~dup_mask].drop(columns="_quality_score")
print(f"passengers: {len(passengers_valid)} passed")

#In passengers build age_band from date_of_birth
bins = [0, 18, 26, 36, 46, 61, 150]
labels = ["<18", "18-25", "26-35", "36-45", "46-60", "60+"]
passengers_silver = passengers_valid.copy()
passengers_silver["age_band"] = pd.cut(passengers_silver["age"],bins=bins,labels=labels,right=False)

#In flights drop exact duplicate rows first
flights_silver = flights_valid.copy()
exact_dupes = flights_silver[flights_silver.duplicated(keep="first")]
quarantine(exact_dupes, "flights","DQ004", "exact duplicate row")
flights_silver = flights_silver.drop_duplicates(keep="first")


#In flights standardize text
for col in ["source", "destination", "airline"]:
    flights_silver[col] = flights_silver[col].astype(str).str.strip().str.upper()
flights_silver["airline"] = flights_silver["airline"].replace({"NAN": None, "UNKNOWN": None})
flights_silver["is_airline_missing"] = flights_silver["airline"].isna()
flights_silver["airline"] = flights_silver["airline"].fillna("Unknown Carrier")


#In flights build flight_key since flight_id repeats across different flights
flights_silver["flight_key"] = (flights_silver["flight_id"] + "_" +flights_silver["departure_time"].dt.strftime("%Y-%m-%d_%H%M%S"))
print("flight_key unique:", flights_silver["flight_key"].is_unique)

route_stats = (flights_silver.groupby(["source", "destination"])["computed_duration_min"].agg(route_mean="mean", route_std="std", route_count="count").reset_index())
flights_silver = flights_silver.merge(route_stats, on=["source", "destination"], how="left")
flights_silver["is_anomaly"] = (
    (flights_silver["route_count"] >= 3) &
    ((flights_silver["computed_duration_min"] - flights_silver["route_mean"]).abs() > 2 * flights_silver["route_std"]))

#In payments flag outlier amounts
payments_silver = payments_valid.copy()
amount_mean = payments_silver["amount"].mean()
amount_std = payments_silver["amount"].std()
payments_silver["is_amount_outlier"] = (payments_silver["amount"] - amount_mean).abs() > 3 * amount_std


#In bookings, already done
bookings_silver = bookings_valid.copy()


print("\nsilver row counts:")
print("  flights   :", len(flights_silver))
print("  bookings  :", len(bookings_silver))
print("  passengers:", len(passengers_silver))
print("  payments  :", len(payments_silver))
print("  overnight :", flights_silver["is_overnight"].sum())
print("  anomaly   :", flights_silver["is_anomaly"].sum())
print("  amount outliers:", payments_silver["is_amount_outlier"].sum())

passengers: 895 passed
flight_key unique: True

silver row counts:
  flights   : 1004
  bookings  : 1000
  passengers: 895
  payments  : 922
  overnight : 122
  anomaly   : 1
  amount outliers: 0


another ambifuity i found is flight_id isn't unique on its own (thats what flight_key is for) but the bookings
table only has flight_id, not a timestamp, so a booking against a reused flight_id
can't always be pinned to one specific flight

In [0]:
match_counts = (bookings_silver.merge(flights_silver[["flight_id", "flight_key"]],on="flight_id",how="left").groupby("booking_id")["flight_key"].nunique())

print("bookings with exactly one flight match:",(match_counts == 1).sum())
print("bookings with multiple flight matches:",(match_counts > 1).sum())
print("bookings with no flight match:",(match_counts == 0).sum())

no_match_booking_ids = match_counts[match_counts == 0].index
ambiguous_booking_ids = match_counts[match_counts > 1].index
quarantine(bookings_silver[bookings_silver["booking_id"].isin(no_match_booking_ids)],"bookings","DQ005","flight_id not found among valid flights")
quarantine(bookings_silver[bookings_silver["booking_id"].isin(ambiguous_booking_ids)],"bookings","DQ008","flight_id matches more than one flight_key, can't resolve without a timestamp")

bad_booking_ids = (set(no_match_booking_ids)| set(ambiguous_booking_ids))
bookings_silver = bookings_silver[ ~bookings_silver["booking_id"].isin(bad_booking_ids)]
print("bookings remaining:", len(bookings_silver))

bookings with exactly one flight match: 997
bookings with multiple flight matches: 2
bookings with no flight match: 1
bookings remaining: 997


Cell 9: PPI Masking

In [0]:
import hashlib

def hash_value(val, salt="asg_airlines_project_salt"):
    if pd.isna(val):
        return None
    return hashlib.sha256((str(val) + salt).encode()).hexdigest()

passengers_pii_vault = passengers_silver[["passenger_id", "first_name", "last_name", "email", "phone", "aadhaar_id", "date_of_birth"]].copy()
passengers_pii_vault["aadhaar_hash"] = passengers_pii_vault["aadhaar_id"].apply(hash_value)
passengers_pii_vault["email_hash"] = passengers_pii_vault["email"].apply(hash_value)
passengers_pii_vault["phone_hash"] = passengers_pii_vault["phone"].apply(hash_value)

passengers_silver_bi = passengers_silver[["passenger_id", "gender", "age_band"]].copy()

bookings_pii_vault = bookings_silver[["booking_id", "passport_number", "emergency_contact_name", "emergency_contact_phone"]].copy()
bookings_silver_bi = bookings_silver[["booking_id", "passenger_id", "flight_id", "booking_date", "status", "seat_number"]].copy()

print("passengers going to Gold:", list(passengers_silver_bi.columns))
print("bookings going to Gold  :", list(bookings_silver_bi.columns))
print("passengers vault rows   :", len(passengers_pii_vault))
print("bookings vault rows     :", len(bookings_pii_vault))

passengers going to Gold: ['passenger_id', 'gender', 'age_band']
bookings going to Gold  : ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'seat_number']
passengers vault rows   : 895
bookings vault rows     : 997


cell 10: Gold

In [0]:
dim_airline = flights_silver[["airline"]].drop_duplicates().reset_index(drop=True)
dim_airline["airline_key"] = dim_airline.index + 1

dim_route = flights_silver[["source", "destination"]].drop_duplicates().reset_index(drop=True)
dim_route["route_key"] = dim_route.index + 1

dim_date = pd.DataFrame({"date": flights_silver["departure_time"].dt.date.unique()})
dim_date["date"] = pd.to_datetime(dim_date["date"])
dim_date["date_key"] = dim_date.index + 1
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["month"] = dim_date["date"].dt.month_name()

dim_passenger = passengers_silver_bi.copy()
dim_passenger["passenger_key"] = range(1, len(dim_passenger) + 1)

print("dim_airline:", len(dim_airline))
print("dim_route  :", len(dim_route))
print("dim_date   :", len(dim_date))
print("dim_passenger:", len(dim_passenger))

dim_airline: 5
dim_route  : 30
dim_date   : 4
dim_passenger: 895


Gold - facts

In [0]:
fact_flights = flights_silver.merge(dim_airline, on="airline", how="left")
fact_flights = fact_flights.merge(dim_route, on=["source", "destination"], how="left")
fact_flights["departure_date"] = fact_flights["departure_time"].dt.floor("D")
fact_flights = fact_flights.merge(dim_date, left_on="departure_date", right_on="date", how="left")
fact_flights = fact_flights[[
    "flight_key", "flight_id", "airline_key", "route_key", "date_key",
    "departure_time", "arrival_time", "computed_duration_min", "is_overnight",
    "is_anomaly", "is_airline_missing"
]].rename(columns={"computed_duration_min": "duration_minutes"})
print("fact_flights:", len(fact_flights))

fact_bookings = bookings_silver_bi.merge(dim_passenger[["passenger_id", "passenger_key"]], on="passenger_id", how="left")
fact_bookings = fact_bookings.merge(flights_silver[["flight_id", "flight_key"]], on="flight_id", how="left")
fact_bookings = fact_bookings[["booking_id", "passenger_key", "flight_key", "status", "booking_date", "seat_number"]]
print("fact_bookings:", len(fact_bookings))

fact_payments = payments_silver[["payment_id", "booking_id", "amount", "payment_method", "is_amount_outlier"]]
print("fact_payments:", len(fact_payments))

fact_flights: 1004
fact_bookings: 997
fact_payments: 922


ALl KPI I have:

Avg duration, route traffic, anomalies, airline distribution, revenue by route, overall anomaly rate

In [0]:
kpi_avg_duration = fact_flights["duration_minutes"].mean()
kpi_route_traffic = (fact_flights.merge(dim_route, on="route_key").groupby(["source", "destination"]).size().sort_values(ascending=False))
kpi_airline_share = (fact_flights.merge(dim_airline, on="airline_key").groupby("airline").size().sort_values(ascending=False))
kpi_anomaly_rate = fact_flights["is_anomaly"].mean() * 100
kpi_revenue_by_route = (fact_bookings.merge(fact_payments, on="booking_id")
                         .merge(fact_flights[["flight_key", "route_key"]], on="flight_key")
                         .merge(dim_route, on="route_key")
                         .groupby(["source", "destination"])["amount"].sum().sort_values(ascending=False))

print("Average flight duration (min):", round(kpi_avg_duration, 1))
print("\nTop routes by traffic:\n", kpi_route_traffic.head())
print("\nFlights by airline:\n", kpi_airline_share)
print("\nAnomaly rate (%):", round(kpi_anomaly_rate, 2))
print("\nTop routes by revenue:\n", kpi_revenue_by_route.head())

Average flight duration (min): 164.5

Top routes by traffic:
 source  destination
BOM     CCU            90
CCU     DEL            72
MAA     BLR            65
BLR     BOM            60
HYD     MAA            57
dtype: int64

Flights by airline:
 airline
INDIGO             249
SPICEJET           235
AIR INDIA          233
VISTARA            218
Unknown Carrier     69
dtype: int64

Anomaly rate (%): 0.1

Top routes by revenue:
 source  destination
BOM     CCU            700135.30
CCU     DEL            582851.37
DEL     HYD            521798.78
BLR     BOM            506069.04
MAA     BLR            492102.05
Name: amount, dtype: float64


In [0]:
# data_quality_metrics
def valid_record_rate(raw_df, valid_df):
    return round(len(valid_df) / len(raw_df) * 100, 2)

def field_completeness(raw_df, required_cols):
    return round(raw_df[required_cols].notna().mean().mean() * 100, 2)

data_quality_metrics = pd.DataFrame([
    {"table": "flights", "run_id": pipeline_run_id,"valid_record_rate": valid_record_rate(flights_raw, flights_silver),
     "field_completeness": field_completeness(flights_raw, ["flight_id", "source", "destination", "departure_time", "arrival_time"])},
    {"table": "bookings", "run_id": pipeline_run_id,"valid_record_rate": valid_record_rate(bookings_raw, bookings_silver),
     "field_completeness": field_completeness(bookings_raw, ["booking_id", "passenger_id", "flight_id", "booking_date", "status"])},
    {"table": "passengers", "run_id": pipeline_run_id,"valid_record_rate": valid_record_rate(passengers_raw, passengers_silver),
     "field_completeness": field_completeness(passengers_raw, ["passenger_id", "first_name", "last_name", "age", "gender"])},
    {"table": "payments", "run_id": pipeline_run_id,"valid_record_rate": valid_record_rate(payments_raw, payments_silver),
     "field_completeness": field_completeness(payments_raw, ["payment_id", "booking_id", "amount", "payment_method"])},
])
data_quality_metrics

,table,run_id,valid_record_rate,field_completeness
0,flights,20260911_062604,98.43,100.00
1,bookings,20260911_062604,99.70,99.10
2,passengers,20260911_062604,86.14,99.81
3,payments,20260911_062604,92.20,98.05


In [0]:
quarantine_table = pd.concat(quarantine_records, ignore_index=True) if quarantine_records else pd.DataFrame()
print("Total quarantined:", len(quarantine_table))
quarantine_table[["source_table", "error_code", "error_reason"]].value_counts()

Total quarantined: 241


source_table  error_code  error_reason                                                                 
passengers    DQ007       aadhaar_id is not a valid 12-digit number                                        114
payments      DQ006       amount is zero or negative or missing                                             78
passengers    DQ004       duplicate passenger_id, kept the more complete record                             30
flights       DQ004       exact duplicate row                                                               15
bookings      DQ008       flight_id matches more than one flight_key, can't resolve without a timestamp      2
              DQ005       flight_id not found among valid flights                                            1
flights       DQ004        arrival_time not after departure_time, even after the overnight fix               1
Name: count, dtype: int64

Cell 10: Exporting for PowerBI

In [0]:
import os

GOLD_PATH = "/Volumes/workspace/default/airline_data/cleaned"
VAULT_PATH = "/Volumes/workspace/default/airline_data/pii_vault"
QUARANTINE_PATH = "/Volumes/workspace/default/airline_data/quarantine"

os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(VAULT_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

tables = {
    "dim_airline": dim_airline,
    "dim_route": dim_route,
    "dim_date": dim_date,
    "dim_passenger": dim_passenger,
    "fact_flights": fact_flights,
    "fact_bookings": fact_bookings,
    "fact_payments": fact_payments,
    "data_quality_metrics": data_quality_metrics,
}

for name, df in tables.items():
    df.to_csv(f"{GOLD_PATH}/{name}.csv", index=False)
    print(f"saved {name}: {len(df)} rows -> {GOLD_PATH}/")

vault_tables = {"passengers_pii_vault": passengers_pii_vault, "bookings_pii_vault": bookings_pii_vault}
for name, df in vault_tables.items():
    df.to_csv(f"{VAULT_PATH}/{name}.csv", index=False)
    print(f"saved {name}: {len(df)} rows -> {VAULT_PATH}/ (restricted)")

quarantine_table.to_csv(f"{QUARANTINE_PATH}/quarantine_table.csv", index=False)
print(f"saved quarantine_table: {len(quarantine_table)} rows -> {QUARANTINE_PATH}/")

saved dim_airline: 5 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved dim_route: 30 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved dim_date: 4 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved dim_passenger: 895 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved fact_flights: 1004 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved fact_bookings: 997 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved fact_payments: 922 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved data_quality_metrics: 4 rows -> /Volumes/workspace/default/airline_data/cleaned/
saved passengers_pii_vault: 895 rows -> /Volumes/workspace/default/airline_data/pii_vault/ (restricted)
saved bookings_pii_vault: 997 rows -> /Volumes/workspace/default/airline_data/pii_vault/ (restricted)
saved quarantine_table: 241 rows -> /Volumes/workspace/default/airline_data/quarantine/
